In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split

In [18]:
X_original = pd.read_csv("imageMNIST.csv", header=None)
y_original = pd.read_csv("labelMNIST.csv")

C:\Users\alexl\AppData\Local\Temp\ipykernel_20988\3943104092.py:1: DtypeWarning: Columns (2,3,4,5,6,7,8,9,15,16,17,18,20,21,22,40,60,80,100,120,140,160,180,200,220,240,260,280,300,320,340,360,379,381,394,395,396,397,398) have mixed types. Specify dtype option on import or set low_memory=False.
  X_original = pd.read_csv("imageMNIST.csv", header=None)


In [19]:
X_original.head()

,0,1,2,3,4,5,6,7,8,9,...,390,391,392,393,394,395,396,397,398,399
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
y_original.head()

,10
0,10
1,10
2,10
3,10
4,10


In [ ]:
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

def gradiente_sigmoide(z):
    s = sigmoide(z)
    return s * (1 - s) #derivada da sigmoide

In [ ]:
def funcao_custo(parametros, tam_entrada, tam_escondida, num_labels, X, y, lambda_regressao):
    parte1, parte2 = np.split(parametros, (tam_entrada + 1) * tam_escondida)
    Theta1 = parte1.reshape(tam_escondida, tam_entrada + 1)
    Theta2 = parte2.reshape(num_labels, tam_escondida + 1)

    m = X.shape[0] #numero de exemplos do treino

    #Forward Propagation
    #camada de entrada
    a1 = np.c_[np.ones((m, 1)), X] #adicionando bias

    #camada oculta
    z2 = a1.dot(Theta1.T)
    a2 = sigmoide(z2)
    a2 = np.c_[np.ones((m, 1)), a2] #adicionado bias

    #camada de saida
    z3 = a2.dot(Theta2.T)
    a3 = sigmoide(z3) #adicionando bias


    y_matriz = np.eye(num_labels)[y] #transformando o y original numa matriz de zeros e uns (One-hot encoding)

    #Custo sem regularizacao
    J = (1 / m) * np.sum(- y_matriz * np.log(a3) - (1 - y_matriz) * np.log(1 - a3))

    #Custo com regularizacao
    J += (lambda_regressao / (2*m)) * (np.sum(np.square(Theta1[:, 1:])) + np.sum(np.square(Theta2[:, 1:])))

    #BackPropagation
    d3 = a3 - y_matriz
    d2 = d3.dot(Theta2[:, 1:]) * gradiente_sigmoide(z2)

    Delta1 = d2.T.dot(a1)
    Delta2 = d3.T.dot(a2)

    Theta1_grad = (1/m) * Delta1
    Theta2_grad = (1/m) * Delta2

    Theta1_grad[:, 1:] += (lambda_regressao / m) * Theta1[:, 1:]
    Theta2_grad[:, 1:] += (lambda_regressao / m) * Theta2[:, 1:]

    grad = np.concatenate([Theta1_grad.ravel(), Theta2_grad.ravel()])
    
    return J, grad

In [ ]:
def gradiente_numerico(funcao_custo, theta, epsilon=1e-4):
    num_grad = np.zeros(theta.shape)
    pertubacao = np.zeros(theta.shape)

    for i in range(theta.size):
        pertubacao[i] = epsilon
        loss1, _ = funcao_custo(theta - pertubacao)
        loss2, _ = funcao_custo(theta + pertubacao)
        num_grad[i] = (loss2 - loss1) / (2 * epsilon)
        pertubacao[i] = 0

    return num_grad

Treinamento com scipy.optimize.minimize

In [ ]:
tam_entrada = 400 #20x20
tam_escondida = 25
num_labels = 10
lambda_reg = 1.0

Theta1_inicial = np.random.rand(tam_escondida, 1 + tam_entrada) * 0.1
Theta2_inicial = np.random.rand(num_labels, 1 + tam_escondida) * 0.1
parametros_iniciais = np.concatenate([Theta1_inicial.flatten(), Theta2_inicial.flatten()])

args = (tam_entrada, [tam_escondida], num_labels, X_original, y_original, lambda_reg)
options = {'maxinter': 400, 'disp':True}

resultado = minimize(fun=funcao_custo, x0=parametros_iniciais, args=args, method='CG', jac=True, options=options)

parametros_opt = resultado.x 

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X_original, y_original, test_size=0.4, random_state=0)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)